# 04 - Feature Engineering

This notebook documents every engineered feature: date features, sales-ratio features, customer/product aggregates, and the monthly aggregated dataset used for forecasting.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import data_loader, feature_engineering

pd.set_option('display.max_columns', 40)

In [2]:
df = data_loader.load_processed_data()
df.shape

(9994, 21)

## 4.1 Date features

From `Order Date`: `Order Year`, `Order Month`, `Order Month Name`, `Order Quarter`, `Order Week`, `Order Day`, `Order Day Of Week`, `Is Weekend`.

In [3]:
df = feature_engineering.add_date_features(df)
df[['Order Date', 'Order Year', 'Order Month', 'Order Quarter',
    'Order Week', 'Order Day Of Week', 'Is Weekend']].head()

,Order Date,Order Year,Order Month,Order Quarter,Order Week,Order Day Of Week,Is Weekend
0,2016-11-08,2016,11,4,45,1,False
1,2016-11-08,2016,11,4,45,1,False
2,2016-06-12,2016,6,2,23,6,True
3,2015-10-11,2015,10,4,41,6,True
4,2015-10-11,2015,10,4,41,6,True


## 4.2 Sales features

`Sales` and `Profit` are already provided by the source dataset, so we do not recompute them. Instead we derive ratio-based features that add analytical value:
- `Discount Amount` = Sales x Discount
- `Profit Margin` = Profit / Sales
- `Average Selling Price` = Sales / Quantity

In [4]:
df = feature_engineering.add_sales_features(df)
df[['Sales', 'Discount', 'Discount Amount', 'Profit', 'Profit Margin',
    'Quantity', 'Average Selling Price']].head()

,Sales,Discount,Discount Amount,Profit,Profit Margin,Quantity,Average Selling Price
0,261.9600,0.00,0.000000,41.9136,0.1600,2,130.9800
1,731.9400,0.00,0.000000,219.5820,0.3000,3,243.9800
2,14.6200,0.00,0.000000,6.8714,0.4700,2,7.3100
3,957.5775,0.45,430.909875,-383.0310,-0.4000,5,191.5155
4,22.3680,0.20,4.473600,2.5164,0.1125,2,11.1840


## 4.3 Customer features

Per-customer aggregates merged back onto every transaction row: total orders, total spending, and average order value.

In [5]:
df = feature_engineering.add_customer_features(df)
df[['Customer ID', 'Customer_Total_Orders', 'Customer_Total_Spending',
    'Customer_Avg_Order_Value']].drop_duplicates('Customer ID').head()

,Customer ID,Customer_Total_Orders,Customer_Total_Spending,Customer_Avg_Order_Value
0,CG-12520,3,1148.7800,229.756000
2,DV-13045,5,1119.4830,124.387000
3,SO-20335,6,2602.5755,173.505033
5,BH-11710,8,6255.3510,260.639625
12,AA-10480,4,1790.5120,149.209333


## 4.4 Product features

Per-product aggregates: total units sold, total revenue, average selling price.

In [6]:
df = feature_engineering.add_product_features(df)
df[['Product ID', 'Product_Total_Units_Sold', 'Product_Total_Revenue',
    'Product_Avg_Selling_Price']].drop_duplicates('Product ID').head()

,Product ID,Product_Total_Units_Sold,Product_Total_Revenue,Product_Avg_Selling_Price
0,FUR-BO-10001798,12,1263.957,315.989250
1,FUR-CH-10000454,51,10637.528,886.460667
2,OFF-LA-10000240,23,153.510,21.930000
3,FUR-TA-10000577,29,7242.768,905.346000
4,OFF-ST-10000760,24,301.968,60.393600


## 4.5 Monthly forecasting dataset (with lag & rolling features)

Transaction-level data is aggregated into a **monthly** time series for sales forecasting. Lag and rolling features (`Prev_Month_Sales`, `Prev_2Month_Sales`, `Rolling_3M_Avg_Sales`) are built using `.shift()` **before** any rolling calculation, so a given month's feature row only ever uses information from *prior* months - this avoids data leakage from the future into the training process.

In [7]:
monthly_df = feature_engineering.build_monthly_forecasting_dataset(df)
monthly_df.shape

(45, 12)

In [8]:
monthly_df.head(10)

,Monthly_Sales,Monthly_Orders,Monthly_Quantity,Monthly_Avg_Discount,Order Date,Year,Month,Quarter,Avg_Order_Value,Prev_Month_Sales,Prev_2Month_Sales,Rolling_3M_Avg_Sales
0,28295.3450,66,536,0.110000,2014-04-01,2014,4,2,428.717348,55691.0090,4519.8920,24815.932000
1,23648.2870,69,466,0.155328,2014-05-01,2014,5,2,342.728797,28295.3450,55691.0090,29502.082000
2,34595.1276,66,521,0.172000,2014-06-01,2014,6,2,524.168600,23648.2870,28295.3450,35878.213667
3,33946.3930,65,550,0.171678,2014-07-01,2014,7,3,522.252200,34595.1276,23648.2870,28846.253200
4,27909.4685,72,609,0.131046,2014-08-01,2014,8,3,387.631507,33946.3930,34595.1276,30729.935867
5,81777.3508,130,1000,0.159963,2014-09-01,2014,9,3,629.056545,27909.4685,33946.3930,32150.329700
6,31453.3930,78,573,0.160063,2014-10-01,2014,10,4,403.248628,81777.3508,27909.4685,47877.737433
7,78628.7167,151,1219,0.151164,2014-11-01,2014,11,4,520.719978,31453.3930,81777.3508,47046.737433
8,69545.6205,141,1079,0.190827,2014-12-01,2014,12,4,493.231351,78628.7167,31453.3930,63953.153500
9,18174.0756,29,236,0.177931,2015-01-01,2015,1,1,626.692262,69545.6205,78628.7167,59875.910067


## Save engineered datasets

In [9]:
df.to_csv('../data/processed/cleaned_sales.csv', index=False)
monthly_df.to_csv('../data/processed/monthly_sales_features.csv', index=False)
print('Transaction-level features:', df.shape)
print('Monthly forecasting dataset:', monthly_df.shape)

Transaction-level features: (9994, 38)
Monthly forecasting dataset: (45, 12)
